In [ ]:
import pickle
tables_data = "./data/dataset_all/all_table_transform_table_corpus.pkl"
with open(tables_data,"rb") as f:
     table_corpus=pickle.load(f)

In [ ]:
print(list(table_corpus["WIKISQL"].keys())[:5])

# Visualize

In [ ]:


import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import torch.nn.functional as F
from ..utils.model import *


def _to_text(x):
    return x if isinstance(x, str) else str(x)

def _l2_normalize_rows(X, eps=1e-12):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (n + eps)

def print_nonzero(t):
    nonzero_vals = t[t != 0]
    print(nonzero_vals)

def check_difference(t: torch.Tensor, adapted_t: torch.Tensor):

    diff = adapted_t - t
    abs_diff = diff.abs()

    out = {
        "max_abs": abs_diff.max().item(),
        "mean_abs": abs_diff.mean().item(),
        "rmse": diff.pow(2).mean().sqrt().item(),
        "l1": abs_diff.sum().item(),
        "l2": diff.norm().item(),
        "num_nonzero_diff": (diff != 0).sum().item(),
        "numel": diff.numel(),
        "fraction_nonzero_diff": ((diff != 0).sum().float() / diff.numel()).item(),
    }
    print(out)

@torch.no_grad()
def plot_one_table_all_from_origin(
    model_name,
    model,
    all_table_corpus,
    dataset="WTQ",
    table_id=None,
    dimension_reduction="tSNE",
    representations=None,
    origin_mode="centroid",     # "centroid" (recommended) or "none"
    normalize_cosine=True,      # recommended
    vis_scale=2.5,
    label_offset=0.06,
    batch_size=64,
    figsize=(11, 7),
    arrow_alpha=0.9,
    point_size=45,
    label_fontsize=11,
    grid_alpha=0.25,
    random_state=0,
):
    """
    Plot all representations as vectors from a common origin.
    "Original" is treated like any other representation.

    origin_mode:
      - "centroid": subtract mean embedding across reps for this table (best)
      - "none": do not center (origin is arbitrary; less interpretable)
    """

    if representations is None:
        representations = [
            #Popular Representation
            'pipe_serialized','token_serialized','space_serialized',
            # Data Representation
            'csv','tsv','html','markdown','latex','dict','json','xml',
            # Structural Transformations 
            'shuffled_rows','shuffled_cols','transpose',
            #Schema and Definition Types
            'mschema','macschema','ddl',
        ]
    category = {
            "Popular Representation": ("centroid_popular",["pipe_serialized", "token_serialized", "space_serialized"],[0,1,2], "red"),
            "Data Representation": ("centroid_data",["csv", "tsv", "html", "markdown", "latex", "dict", "json", "xml"],[3,4,5,6,7,8,9,10],"blue"),
            "Structural Transformations": ("centroid_data",["shuffled_rows", "shuffled_cols", "transpose"],[11,12,12],"green"),
            "Schema and Definition Types": ("centroid_schema",["mschema", "macschema", "ddl"],[14,15,16],"purple"),
    }
    table_corpus = all_table_corpus[dataset]
    if table_id is None:
        table_id = next(iter(table_corpus.keys()))

    # ---- Collect texts
    rep_texts, rep_names, missing = [], [], []
    for rep in representations:
        if rep not in table_corpus[table_id]:
            missing.append(rep)
            continue
        rep_texts.append(_to_text(table_corpus[table_id][rep]))
        rep_names.append(rep)

    if len(rep_texts) < 2:
        raise ValueError(f"Need >=2 reps. Got {len(rep_texts)}. Missing={missing}")

    # ---- Embed
    embs = encode_texts_in_batches(
        model_name,
        model,
        rep_texts,
        instruction="",
        batch_size=batch_size,
    )
    
    if isinstance(embs, np.ndarray):
        embs = torch.from_numpy(embs)
    if model_name == "splade":
        embs = embs.to_dense()
    embs = embs.float().cpu().numpy()  # (R, D)

    # ---- Center to make a meaningful common origin
    X = embs.copy()

    # ===================== NEW: colors + category centroids (+ centroid_all) =====================
    # Build per-representation colors (default gray if not in any category)
    rep_colors = ["gray"] * len(rep_names)
    for _, (_, _, idxs, color) in category.items():
        for j in idxs:
            if 0 <= j < len(rep_names):
                rep_colors[j] = color

    # Compute category centroids from X using provided indices (only valid indices)
    centroid_names = []
    centroid_vecs = []
    centroid_colors = []

    for _, (cent_name, _, idxs, color) in category.items():
        valid = [j for j in idxs if 0 <= j < X.shape[0]]
        if len(valid) == 0:
            continue
        centroid_vecs.append(X[valid].mean(axis=0))
        centroid_names.append(cent_name)
        centroid_colors.append(color)

    # centroid_all over all available reps
    centroid_all = X.mean(axis=0)
    centroid_vecs.append(centroid_all)
    centroid_names.append("centroid_all")
    centroid_colors.append("black")  # choose a neutral color for "all"

    # Augment X / names / colors with centroid vectors
    if len(centroid_vecs) > 0:
        X = np.vstack([X, np.stack(centroid_vecs, axis=0)])
        rep_names = rep_names + centroid_names
        rep_colors = rep_colors + centroid_colors


    if origin_mode.lower() == "centroid":
        mu = X.mean(axis=0, keepdims=True)
        X = X - mu
    elif origin_mode.lower() == "none":
        pass
    else:
        raise ValueError("origin_mode must be 'centroid' or 'none'")

    # ---- Cosine normalization (direction-focused)
    if normalize_cosine:
        X = _l2_normalize_rows(X)

    if dimension_reduction == "PCA":
        # ---- PCA to 2D
        Z = PCA(n_components=2, random_state=random_state).fit_transform(X)  # (R, 2)
    elif dimension_reduction == "tSNE":
        n = X.shape[0]  # 17
        perp = min(5, n - 1)
        X_50 = PCA(n_components=min(50, X.shape[0]-1), random_state=random_state).fit_transform(X)

        Z = TSNE(
            n_components=2,
            perplexity=perp,
            init="pca",
            learning_rate="auto",
            random_state=random_state,
        ).fit_transform(X_50)


    # ---- Visual scaling
    Z_vis = Z * float(vis_scale)

    # ---- Plot
    fig, ax = plt.subplots(figsize=figsize)
    

    # Arrow head sizing
    span = max(Z_vis[:,0].max() - Z_vis[:,0].min(), Z_vis[:,1].max() - Z_vis[:,1].min())
    head_w = 0.02 * (span + 1e-9)
    head_l = 0.03 * (span + 1e-9)

    # Draw origin point
    ax.scatter([0], [0], s=point_size)
    ax.text(0, 0, " origin", fontsize=label_fontsize, va="center")

    # Draw arrows from (0,0) to each rep
    for i, name in enumerate(rep_names):
        xi, yi = Z_vis[i]
        if "centroid" in name:
            ax.scatter(xi, yi, s=point_size+5,color="black",marker="x")
        else:
            ax.scatter(xi, yi, s=point_size,color=rep_colors[i])
        ax.arrow(
            0.0, 0.0, xi, yi,
            length_includes_head=True,
            head_width=head_w,
            head_length=head_l,
            alpha=arrow_alpha,
            linewidth=1.6,
            color="black"  if "centroid" in name else rep_colors[i] ,
            hatch="o"  if "centroid" in name else "x" 
        )

        # label slightly offset along its direction
        v = np.array([xi, yi], dtype=float)
        n = np.linalg.norm(v) + 1e-9
        ux, uy = v / n
        ax.text(
            xi + label_offset * ux,
            yi + label_offset * uy,
            name,
            fontsize=label_fontsize,
            va="center",
            ha="left"
        )

    ax.set_title(
        f"Representation embeddings from a common origin (table_id={table_id})\n"
        f"origin={origin_mode} | cosine_norm={normalize_cosine} | scale={vis_scale}"
    )
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")
    ax.axhline(0, linewidth=0.6)
    ax.axvline(0, linewidth=0.6)
    ax.grid(True, linewidth=0.5, alpha=grid_alpha)

    # Padding so labels don’t clip
    pad = 0.12 * (span + 1e-9)
    ax.set_xlim(Z_vis[:,0].min() - pad, Z_vis[:,0].max() + pad)
    ax.set_ylim(Z_vis[:,1].min() - pad, Z_vis[:,1].max() + pad)

    #plt.tight_layout()

    if missing:
        print(f"[plot] Missing reps for table_id={table_id}: {missing}")

    return fig, ax, {"table_id": table_id, "rep_names": rep_names, "Z": Z, "Z_vis": Z_vis}

@torch.no_grad()
def plot_one_table_all_with_adapter_comparison(
    model_name,
    model,
    adapter,
    all_table_corpus,
    dataset="WTQ",
    table_id=None,
    representations=None,
    dimension_reduction="PCA",   # PCA recommended for comparing movement
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.5,
    label_offset=0.06,
    batch_size=64,
    adapter_device="cuda",
    figsize=(8, 6),
    point_size=55,
    label_fontsize=10,
    grid_alpha=0.25,
    random_state=0,
    draw_labels=True,
    draw_shift_arrows=True,
):
    """
    Compare representation embeddings before vs after adapter.

    Important:
    - Original and adapted embeddings are projected together in ONE shared 2D space.
    - This makes displacement visually meaningful.
    """

    if representations is None:
        representations = [
            "pipe_serialized", "token_serialized", "space_serialized",
            "csv", "tsv", "html", "markdown", "latex", "dict", "json", "xml",
            "shuffled_rows", "shuffled_cols", "transpose",
            "mschema", "macschema", "ddl",
        ]

    category = {
        "Popular Representation": (
            ["pipe_serialized", "token_serialized", "space_serialized"],
            "red",
        ),
        "Data Representation": (
            ["csv", "tsv", "html", "markdown", "latex", "dict", "json", "xml"],
            "blue",
        ),
        "Structural Transformations": (
            ["shuffled_rows", "shuffled_cols", "transpose"],
            "green",
        ),
        "Schema and Definition Types": (
            ["mschema", "macschema", "ddl"],
            "purple",
        ),
    }

    table_corpus = all_table_corpus[dataset]
    if table_id is None:
        table_id = next(iter(table_corpus.keys()))

    # Collect texts
    rep_texts, rep_names, missing = [], [], []
    for rep in representations:
        if rep not in table_corpus[table_id]:
            missing.append(rep)
            continue
        rep_texts.append(_to_text(table_corpus[table_id][rep]))
        rep_names.append(rep)

    if len(rep_texts) < 2:
        raise ValueError(f"Need >= 2 reps. Got {len(rep_texts)}. Missing={missing}")

    # Map rep -> color
    rep_colors = []
    for rep in rep_names:
        color = "gray"
        for _, (members, c) in category.items():
            if rep in members:
                color = c
                break
        rep_colors.append(color)

    # Encode original embeddings
    embs = encode_texts_in_batches(
        model_name,
        model,
        rep_texts,
        instruction="",
        batch_size=batch_size,
    )

    if isinstance(embs, np.ndarray):
        embs_t = torch.from_numpy(embs)
    else:
        embs_t = embs

    if model_name == "splade":
        embs_t = embs_t.to_dense()

    embs_t = embs_t.float().cpu()
    original_embs = embs_t.numpy().copy()

    # Optional normalization before adapter
    if normalize_before_adapter:
        original_embs = _l2_normalize_rows(original_embs)

    # Apply adapter
    adapter.eval()
    adapter = adapter.to(adapter_device)

    t = torch.from_numpy(original_embs).to(adapter_device)
    print("Original:")
    print_nonzero(t)
    adapted_t = adapter(t)
    print("Adapted:")
    print_nonzero(adapted_t)
    print("DIFFERENCE")
    check_difference(t, adapted_t)
    if normalize_after_adapter:
        adapted_t = F.normalize(adapted_t, p=2, dim=-1)
    adapted_embs = adapted_t.detach().cpu().numpy().astype(np.float32, copy=False)
    # Joint space for fair comparison
    # Add centroid_all for original and adapted
    centroid_all_orig = original_embs.mean(axis=0, keepdims=True)   # (1, D)
    centroid_all_adapt = adapted_embs.mean(axis=0, keepdims=True)   # (1, D)

    # Joint space for fair comparison
    X_joint = np.vstack([
        original_embs,
        adapted_embs,
        centroid_all_orig,
        centroid_all_adapt,
    ])   # shape (2R + 2, D)



    # Dimensionality reduction in one shared space
    if dimension_reduction == "PCA":
        reducer = PCA(n_components=2, random_state=random_state)
        Z_joint = reducer.fit_transform(X_joint)

    elif dimension_reduction == "tSNE":
        n = X_joint.shape[0]
        perp = min(5, n - 1)
        X_pre = PCA(
            n_components=min(50, X_joint.shape[0] - 1, X_joint.shape[1]),
            random_state=random_state,
        ).fit_transform(X_joint)

        Z_joint = TSNE(
            n_components=2,
            perplexity=perp,
            init="pca",
            learning_rate="auto",
            random_state=random_state,
        ).fit_transform(X_pre)
    else:
        raise ValueError("dimension_reduction must be 'PCA' or 'tSNE'")

    R = len(rep_names)
    Z_orig = Z_joint[:R] * float(vis_scale)
    Z_adapt = Z_joint[R:2*R] * float(vis_scale)
    Z_centroid_orig = Z_joint[2*R] * float(vis_scale)
    Z_centroid_adapt = Z_joint[2*R + 1] * float(vis_scale)

    # Plot
    fig, ax = plt.subplots(figsize=figsize)

    # Origin marker
    ax.scatter([0], [0], s=40, color="black")
    ax.text(0, 0, " origin", fontsize=label_fontsize, va="center")

    # Plot each representation before/after
    for i, name in enumerate(rep_names):
        x0, y0 = Z_orig[i]
        x1, y1 = Z_adapt[i]
        c = rep_colors[i]

        # Original
        ax.scatter(x0, y0, s=point_size, color=c, marker="o", alpha=0.9)

        # Adapted
        ax.scatter(x1, y1, s=point_size, color=c, marker="X", alpha=0.95)

        # Movement line / arrow
        if draw_shift_arrows:
            ax.annotate(
                "",
                xy=(x1, y1),
                xytext=(x0, y0),
                arrowprops=dict(
                    arrowstyle="->",
                    color=c,
                    lw=1.5,
                    alpha=0.85,
                    linestyle="--",
                ),
            )

        # Labels
        if draw_labels:
            v0 = np.array([x0, y0], dtype=float)
            n0 = np.linalg.norm(v0) + 1e-9
            u0 = v0 / n0

            v1 = np.array([x1, y1], dtype=float)
            n1 = np.linalg.norm(v1) + 1e-9
            u1 = v1 / n1

            ax.text(
                x0 + label_offset * u0[0],
                y0 + label_offset * u0[1],
                f"{name}_orig",
                fontsize=label_fontsize,
                ha="left",
                va="center",
            )
            ax.text(
                x1 + label_offset * u1[0],
                y1 + label_offset * u1[1],
                f"{name}_adapt",
                fontsize=label_fontsize,
                ha="left",
                va="center",
            )

    # Plot centroid_all with a distinct marker
    ax.scatter(
        Z_centroid_orig[0], Z_centroid_orig[1],
        s=point_size + 80,
        color="black",
        marker="D",
        alpha=0.9,
        label="centroid_all (without adapter)"
    )

    ax.scatter(
        Z_centroid_adapt[0], Z_centroid_adapt[1],
        s=point_size + 100,
        color="black",
        marker="P",
        alpha=0.95,
        label="centroid_all (with adapter)"
    )

    ax.annotate(
        "",
        xy=(Z_centroid_adapt[0], Z_centroid_adapt[1]),
        xytext=(Z_centroid_orig[0], Z_centroid_orig[1]),
        arrowprops=dict(
            arrowstyle="->",
            color="black",
            lw=2.0,
            alpha=0.9,
            linestyle="--",
        ),
    ) 

    if draw_labels:

        ax.text(
            Z_centroid_orig[0] + label_offset,
            Z_centroid_orig[1] + label_offset,
            "centroid_all_orig",
            fontsize=label_fontsize,
            ha="left",
            va="center",
            fontweight="bold",
        )
        ax.text(
            Z_centroid_adapt[0] + label_offset,
            Z_centroid_adapt[1] + label_offset,
            "centroid_all_adapt",
            fontsize=label_fontsize,
            ha="left",
            va="center",
            fontweight="bold",
        )

    # Legend proxies
    ax.scatter([], [], color="black", marker="o", label="Without adapter")
    ax.scatter([], [], color="black", marker="X", label="With adapter")
    #ax.legend()

    '''ax.set_title(
        f"Embedding shift by adapter (table_id={table_id})\n"
        f"reduction={dimension_reduction} "
    )'''
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")
    ax.axhline(0, linewidth=0.6, color="black", alpha=0.6)
    ax.axvline(0, linewidth=0.6, color="black", alpha=0.6)
    ax.grid(True, linewidth=0.5, alpha=grid_alpha)

    # Bounds
    Z_all = np.vstack([Z_orig, Z_adapt])
    span = max(
        Z_all[:, 0].max() - Z_all[:, 0].min(),
        Z_all[:, 1].max() - Z_all[:, 1].min()
    )
    pad = 0.12 * (span + 1e-9)
    ax.set_xlim(Z_all[:, 0].min() - pad, Z_all[:, 0].max() + pad)
    ax.set_ylim(Z_all[:, 1].min() - pad, Z_all[:, 1].max() + pad)

    if missing:
        print(f"[plot] Missing reps for table_id={table_id}: {missing}")
    # ── Save projected data to CSV ──────────────────────────────────────
    rows = []

    # Rep points (original)
    for i, name in enumerate(rep_names):
        rows.append({
            "point_type":   "rep_orig",
            "table_id":     table_id,
            "rep":          name,
            "color":        rep_colors[i],
            "dim1":         float(Z_orig[i, 0]),
            "dim2":         float(Z_orig[i, 1]),
        })

    # Rep points (adapted)
    for i, name in enumerate(rep_names):
        rows.append({
            "point_type":   "rep_adapt",
            "table_id":     table_id,
            "rep":          name,
            "color":        rep_colors[i],
            "dim1":         float(Z_adapt[i, 0]),
            "dim2":         float(Z_adapt[i, 1]),
        })

    # Global centroid (original)
    rows.append({
        "point_type":   "centroid_orig",
        "table_id":     table_id,
        "rep":          "",
        "color":        "black",
        "dim1":         float(Z_centroid_orig[0]),
        "dim2":         float(Z_centroid_orig[1]),
    })

    # Global centroid (adapted)
    rows.append({
        "point_type":   "centroid_adapt",
        "table_id":     table_id,
        "rep":          "",
        "color":        "black",
        "dim1":         float(Z_centroid_adapt[0]),
        "dim2":         float(Z_centroid_adapt[1]),
    })

    csv_path = (
        f"./figure/adapter_comparison"
        f"_{dataset}"
        f"_{model_name}"
        f"_{dimension_reduction}"
        f"_table{table_id.replace('/','-')}"
        f".csv"
    )
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"[plot] Projected data saved to: {csv_path}")
    # ────────────────────────────────────────────────────────────────────


    return fig, ax, {
        "table_id": table_id,
        "rep_names": rep_names,
        "original_embs": original_embs,
        "adapted_embs": adapted_embs,
        "Z_orig": Z_orig,
        "Z_adapt": Z_adapt,
        "shift": Z_adapt - Z_orig,
    }

@torch.no_grad()
def plot_random_tables_all_with_adapter_comparison(
        model_name,
        model,
        adapter,
        all_table_corpus,
        dataset="WTQ",
        table_ids=None,                # optional explicit list; if None, sample
        num_tables=10,
        representations=None,
        dimension_reduction="PCA",
        normalize_before_adapter=False,
        normalize_after_adapter=True,
        vis_scale=2.5,
        label_offset=0.06,
        batch_size=64,                 # embedding batch size (texts)
        adapter_device="cuda",
        adapter_batch_size=8192,       # adapter forward batch size (vectors)
        figsize=(8, 6),
        point_size=55,
        label_fontsize=10,
        grid_alpha=0.25,
        random_state=0,
        draw_labels=True,
        draw_shift_arrows=True,
        draw_centroid_shift_arrows=True,
    ):
        """
        Sample num_tables tables, embed each representation for each table,
        apply adapter, project ALL points together in ONE shared 2D space.

        Centroids:
        - One centroid per table.
        - Each centroid is the mean of that table's representation embeddings only.
        - We plot both centroid_orig and centroid_adapt per table.
        """

        if representations is None:
            representations = [
                "pipe_serialized", "token_serialized", "space_serialized",
                "csv", "tsv", "html", "markdown", "latex", "dict", "json", "xml",
                "shuffled_rows", "shuffled_cols", "transpose",
                "mschema", "macschema", "ddl",
            ]

        category = {
            "Popular Representation": (["pipe_serialized", "token_serialized", "space_serialized"], "red"),
            "Data Representation": (["csv", "tsv", "html", "markdown", "latex", "dict", "json", "xml"], "blue"),
            "Structural Transformations": (["shuffled_rows", "shuffled_cols", "transpose"], "green"),
            "Schema and Definition Types": (["mschema", "macschema", "ddl"], "purple"),
        }

        table_corpus = all_table_corpus[dataset]
        all_ids = list(table_corpus.keys())
        if len(all_ids) == 0:
            raise ValueError(f"No tables found for dataset={dataset}")

        rng = np.random.default_rng(random_state)

        if table_ids is None:
            k = min(int(num_tables), len(all_ids))
            sampled_ids = list(rng.choice(all_ids, size=k, replace=False))
        else:
            sampled_ids = list(table_ids)
            if len(sampled_ids) == 0:
                raise ValueError("table_ids provided but empty")

        # Build rep -> color (fixed per representation)
        rep_to_color = {}
        for rep in representations:
            color = "gray"
            for _, (members, c) in category.items():
                if rep in members:
                    color = c
                    break
            rep_to_color[rep] = color

        # Collect texts across sampled tables and track per-table index ranges
        all_texts = []
        all_rep_names = []        # rep name per point
        all_missing = {}          # table_id -> missing reps
        table_point_indices = {}  # table_id -> list of indices into all_texts for that table

        for tid in sampled_ids:
            idxs = []
            missing = []
            for rep in representations:
                if rep not in table_corpus[tid]:
                    missing.append(rep)
                    continue
                idxs.append(len(all_texts))
                all_texts.append(_to_text(table_corpus[tid][rep]))
                all_rep_names.append(rep)
            table_point_indices[tid] = idxs
            if missing:
                all_missing[tid] = missing

        if len(all_texts) < 2:
            raise ValueError("Need >= 2 total representations across sampled tables")

        # Embed originals for all collected texts
        embs = encode_texts_in_batches(
            model_name,
            model,
            all_texts,
            instruction="",
            batch_size=batch_size,
        )

        if isinstance(embs, np.ndarray):
            embs_t = torch.from_numpy(embs)
        else:
            embs_t = embs

        if model_name == "splade":
            embs_t = embs_t.to_dense()

        embs_t = embs_t.float().cpu()
        original_embs = embs_t.numpy().copy()

        if normalize_before_adapter:
            original_embs = _l2_normalize_rows(original_embs)

        # Apply adapter in batches over vectors
        adapter.eval()
        adapter = adapter.to(adapter_device)

        adapted_embs = np.zeros_like(original_embs, dtype=np.float32)
        for start in range(0, original_embs.shape[0], adapter_batch_size):
            end = min(start + adapter_batch_size, original_embs.shape[0])
            t = torch.from_numpy(original_embs[start:end]).to(adapter_device)
            z = adapter(t)
            if normalize_after_adapter:
                z = F.normalize(z, p=2, dim=-1)
            adapted_embs[start:end] = z.detach().cpu().numpy().astype(np.float32, copy=False)

        # Compute per-table centroids (orig and adapted), one per table
        centroid_table_ids = []
        centroids_orig = []
        centroids_adapt = []

        for tid in sampled_ids:
            idxs = table_point_indices.get(tid, [])
            if len(idxs) == 0:
                continue
            centroids_orig.append(original_embs[idxs].mean(axis=0))
            centroids_adapt.append(adapted_embs[idxs].mean(axis=0))
            centroid_table_ids.append(tid)

        if len(centroids_orig) == 0:
            raise ValueError("No table centroids computed (no representations available in sampled tables).")

        centroids_orig = np.stack(centroids_orig, axis=0)   # (T, D)
        centroids_adapt = np.stack(centroids_adapt, axis=0) # (T, D)

        # Joint space: reps + adapted reps + per-table centroids + adapted centroids
        X_joint = np.vstack([
            original_embs,
            adapted_embs,
            centroids_orig,
            centroids_adapt,
        ])  # (2N + 2T, D)

        # Dimensionality reduction in one shared space
        if dimension_reduction == "PCA":
            reducer = PCA(n_components=2, random_state=random_state)
            Z_joint = reducer.fit_transform(X_joint)
        elif dimension_reduction == "tSNE":
            n = X_joint.shape[0]
            perp = min(30, max(5, (n - 1) // 10), n - 1)
            X_pre = PCA(
                n_components=min(50, X_joint.shape[0] - 1, X_joint.shape[1]),
                random_state=random_state,
            ).fit_transform(X_joint)
            Z_joint = TSNE(
                n_components=2,
                perplexity=perp,
                init="pca",
                learning_rate="auto",
                random_state=random_state,
            ).fit_transform(X_pre)
        else:
            raise ValueError("dimension_reduction must be 'PCA' or 'tSNE'")

        N = len(all_rep_names)
        T = len(centroid_table_ids)

        Z_orig = Z_joint[:N] * float(vis_scale)
        Z_adapt = Z_joint[N:2*N] * float(vis_scale)
        Z_centroid_orig = Z_joint[2*N:2*N + T] * float(vis_scale)
        Z_centroid_adapt = Z_joint[2*N + T:2*N + 2*T] * float(vis_scale)

        # Plot
        fig, ax = plt.subplots(figsize=figsize)

        ax.scatter([0], [0], s=40, color="black")
        ax.text(0, 0, " origin", fontsize=label_fontsize, va="center")

        # Plot all rep points (aggregate, no table differentiation)
        labeled_orig = set()
        labeled_adapt = set()

        for i, rep in enumerate(all_rep_names):
            x0, y0 = Z_orig[i]
            x1, y1 = Z_adapt[i]
            c = rep_to_color.get(rep, "gray")

            ax.scatter(x0, y0, s=point_size, color=c, marker="o", alpha=0.25)
            ax.scatter(x1, y1, s=point_size, color=c, marker="X", alpha=0.25)

            if draw_shift_arrows:
                ax.annotate(
                    "",
                    xy=(x1, y1),
                    xytext=(x0, y0),
                    arrowprops=dict(
                        arrowstyle="->",
                        color=c,
                        lw=1.0,
                        alpha=0.15,
                        linestyle="--",
                    ),
                )

            if draw_labels:
                # label each representation once
                if rep not in labeled_orig:
                    v0 = np.array([x0, y0], dtype=float)
                    u0 = v0 / (np.linalg.norm(v0) + 1e-9)
                    ax.text(
                        x0 + label_offset * u0[0],
                        y0 + label_offset * u0[1],
                        f"{rep}_orig",
                        fontsize=label_fontsize,
                        ha="left",
                        va="center",
                        alpha=0.9,
                    )
                    labeled_orig.add(rep)

                if rep not in labeled_adapt:
                    v1 = np.array([x1, y1], dtype=float)
                    u1 = v1 / (np.linalg.norm(v1) + 1e-9)
                    ax.text(
                        x1 + label_offset * u1[0],
                        y1 + label_offset * u1[1],
                        f"{rep}_adapt",
                        fontsize=label_fontsize,
                        ha="left",
                        va="center",
                        alpha=0.9,
                    )
                    labeled_adapt.add(rep)

        # Plot per-table centroids with distinct markers
        # Orig centroids
        ax.scatter(
            Z_centroid_orig[:, 0], Z_centroid_orig[:, 1],
            s=point_size + 110,
            color="black",
            marker="D",
            alpha=0.75,
            label="table centroid (without adapter)",
        )
        # Adapt centroids
        ax.scatter(
            Z_centroid_adapt[:, 0], Z_centroid_adapt[:, 1],
            s=point_size + 130,
            color="black",
            marker="P",
            alpha=0.80,
            label="table centroid (with adapter)",
        )
        # Label each centroid pair with its table_id
        for i, tid in enumerate(centroid_table_ids):
            # Label on the adapted centroid
            ax.text(
                Z_centroid_orig[i, 0] + label_offset,
                Z_centroid_orig[i, 1] + label_offset,
                str(tid),
                fontsize=label_fontsize - 1,
                ha="left",
                va="center",
                color="black",
                alpha=0.95,
                fontweight="bold",
            )
        if draw_centroid_shift_arrows:
            for i in range(T):
                ax.annotate(
                    "",
                    xy=(Z_centroid_adapt[i, 0], Z_centroid_adapt[i, 1]),
                    xytext=(Z_centroid_orig[i, 0], Z_centroid_orig[i, 1]),
                    arrowprops=dict(
                        arrowstyle="->",
                        color="black",
                        lw=1.8,
                        alpha=0.55,
                        linestyle="--",
                    ),
                )

        # Legend proxies for rep markers
        ax.scatter([], [], color="black", marker="o", label="Without adapter")
        ax.scatter([], [], color="black", marker="X", label="With adapter")
        #ax.legend()

        '''ax.set_title(
            f"Embedding shift by adapter across {len(sampled_ids)} tables (dataset={dataset})\n"
            f"reduction={dimension_reduction}"
        )'''
        ax.set_xlabel("Dim 1")
        ax.set_ylabel("Dim 2")
        ax.axhline(0, linewidth=0.6, color="black", alpha=0.6)
        ax.axvline(0, linewidth=0.6, color="black", alpha=0.6)
        ax.grid(True, linewidth=0.5, alpha=grid_alpha)

        Z_all = np.vstack([Z_orig, Z_adapt, Z_centroid_orig, Z_centroid_adapt])
        span = max(
            Z_all[:, 0].max() - Z_all[:, 0].min(),
            Z_all[:, 1].max() - Z_all[:, 1].min(),
        )
        pad = 0.12 * (span + 1e-9)
        ax.set_xlim(Z_all[:, 0].min() - pad, Z_all[:, 0].max() + pad)
        ax.set_ylim(Z_all[:, 1].min() - pad, Z_all[:, 1].max() + pad)

        if all_missing:
            print(f"[plot] Missing reps in {len(all_missing)}/{len(sampled_ids)} sampled tables.")

        # ── Save projected data to CSV ──────────────────────────────────────
        # Build a flat record for every plotted point so the figure can be
        # fully reconstructed from the CSV without re-running the model.

        # 1. Map each raw point back to its table_id
        point_table_ids = [""] * N
        for tid, idxs in table_point_indices.items():
            for idx in idxs:
                point_table_ids[idx] = tid

        rows = []

        # Rep points (original)
        for i, rep in enumerate(all_rep_names):
            rows.append({
                "point_type":   "rep_orig",
                "table_id":     point_table_ids[i],
                "rep":          rep,
                "color":        rep_to_color.get(rep, "gray"),
                "dim1":         float(Z_orig[i, 0]),
                "dim2":         float(Z_orig[i, 1]),
            })

        # Rep points (adapted)
        for i, rep in enumerate(all_rep_names):
            rows.append({
                "point_type":   "rep_adapt",
                "table_id":     point_table_ids[i],
                "rep":          rep,
                "color":        rep_to_color.get(rep, "gray"),
                "dim1":         float(Z_adapt[i, 0]),
                "dim2":         float(Z_adapt[i, 1]),
            })

        # Per-table centroids (original)
        for i, tid in enumerate(centroid_table_ids):
            rows.append({
                "point_type":   "centroid_orig",
                "table_id":     tid,
                "rep":          "",
                "color":        "black",
                "dim1":         float(Z_centroid_orig[i, 0]),
                "dim2":         float(Z_centroid_orig[i, 1]),
            })

        # Per-table centroids (adapted)
        for i, tid in enumerate(centroid_table_ids):
            rows.append({
                "point_type":   "centroid_adapt",
                "table_id":     tid,
                "rep":          "",
                "color":        "black",
                "dim1":         float(Z_centroid_adapt[i, 0]),
                "dim2":         float(Z_centroid_adapt[i, 1]),
            })

        csv_path = (
            f"./figure/adapter_comparison_10"
            f"_{dataset}"
            f"_{model_name}"
            f"_{dimension_reduction}"
            f"_{len(sampled_ids)}_tables"
            f".csv"
        )
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        print(f"[plot] Projected data saved to: {csv_path}")
        # ────────────────────────────────────────────────────────────────────

        return fig, ax

# Model Selection

In [ ]:
from transformers import AutoModel
from accelerate import Accelerator
import torch
from test_universal_adapter import load_adapter

import os
os.environ["CUDA_VISIBLE_DEVICES"]="5"
model_name="splade"

accelerator = Accelerator()

# get ReasonIR model
model = get_model(accelerator,model_name=model_name)
adapter_path = f"./data/2026-03-01/centroid_adapter/{model_name}/adapter.pt" #_subset_dataset
adapter, adapter_dim, _ = load_adapter(adapter_path, device="cuda")


# WTQ Table

## With Adapter

In [ ]:
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="WTQ",
    table_id='csv/200-csv/0',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
    draw_labels=False
)
plt.show()
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="WTQ",
    table_id='csv/200-csv/1',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
    draw_labels=False
)
plt.show()
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="WTQ",
    table_id='csv/200-csv/10',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
    draw_labels=False
)
plt.show()
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="WTQ",
    table_id='csv/200-csv/11',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
    draw_labels=False
)
plt.show()


## Multi Table

In [ ]:
tableid = ["csv/204-csv/598","csv/202-csv/102","csv/200-csv/48","csv/203-csv/823","csv/203-csv/448","csv/202-csv/226","csv/204-csv/267","csv/204-csv/66","csv/203-csv/378","csv/203-csv/203"]
for tid in ["csv/203-csv/448","csv/204-csv/66"]:
  print(tid)
  print(table_corpus["WTQ"][tid]["df_string"])


In [ ]:
fig, ax= plot_random_tables_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="WTQ",
    num_tables=10,
    table_ids=tableid,
    dimension_reduction="PCA",
    draw_labels=False,
)
plt.show()

# WIKISQL

## With Adapter

In [ ]:
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="WIKISQL",
    table_id='1-10015132-16',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="WIKISQL",
    table_id= '1-10083598-1',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="WIKISQL",
    table_id= '1-1013129-2',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="WIKISQL",
    table_id= '1-1013129-3',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()


## MultiTable

In [ ]:
fig, ax= plot_random_tables_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="WIKISQL",
    num_tables=10,
    dimension_reduction="PCA",
)
plt.show()

# NQ

## With Adapter

In [ ]:
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="NQ",
    table_id='Lesley Joseph_A1D55A57012E3362',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="NQ",
    table_id='List of India national cricket captains_DF24C9CC57542D09',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="NQ",
    table_id= 'Infinity on High_C66E938E06D39ECB',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()
fig, ax, out = plot_one_table_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="NQ",
    table_id= 'Hydrogen sulfide_54EB24287C466E6E',
    dimension_reduction="PCA",
    normalize_before_adapter=False,
    normalize_after_adapter=True,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()

## MultiTable

In [ ]:
fig, ax= plot_random_tables_all_with_adapter_comparison(
    model_name=model_name,
    model=model,
    adapter=adapter,
    all_table_corpus=table_corpus,
    dataset="NQ",
    num_tables=10,
    dimension_reduction="PCA",
)
plt.show()